# 第6章 数の表を一気に扱う：テンソル

for文を卒業して、420万マスの表を一瞬で計算する

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備


In [ ]:
import re, math, time, urllib.request
from collections import Counter
import torch

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
print("語彙の数：", V)

## テンソル入門：数を並べた箱


In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
print(a + b)
print(a * b)
print(a * 2)
print(a.sum())

In [ ]:
M = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
print(M.shape)
print(M[0])       # 0行目
print(M[:, 1])    # 1列目（全部の行の、1番目）
print(M[1, 2])    # 1行目の2列目

## bigramの表を作る


In [ ]:
n = int(len(text) * 0.9)
train_text, val_text = text[:n], text[n:]

pairs = Counter(zip(train_text, train_text[1:]))
N = torch.zeros((V, V))
for (a, b), count in pairs.items():
    N[stoi[a], stoi[b]] = count

print(N.shape)
print("「先」→「生」の回数：", N[stoi["先"], stoi["生"]].item())
print("表のマスの数：", N.numel(), " うち0でないマス：", (N > 0).sum().item())

## 回数を確率に変える：行ごとの合計とブロードキャスト


In [ ]:
k = 0.01
row_sums = (N + k).sum(dim=1, keepdim=True)
print(row_sums.shape)

P = (N + k) / row_sums
print(P.shape)
print("各行の合計（最初の5行）：", P.sum(dim=1)[:5])

## 損失を一瞬で計算する


In [ ]:
val_ids = torch.tensor([stoi[c] for c in val_text])
xs, ys = val_ids[:-1], val_ids[1:]

start = time.time()
probs = P[xs, ys]
loss = -probs.log().mean()
print(f"検証データの損失：{loss.item():.3f}（{time.time() - start:.4f}秒）")

## 表を眺める


In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

look = ["私", "先", "生", "は", "の", "に", "を", "た", "。", "、", "奥", "さ", "ん", "Ｋ"]
idx = [stoi[c] for c in look]
sub = P[idx][:, idx]

plt.figure(figsize=(6, 5))
plt.imshow(sub, cmap="Blues")
plt.xticks(range(len(look)), look)
plt.yticks(range(len(look)), look)
plt.xlabel("次の文字")
plt.ylabel("今の文字")
plt.colorbar(label="確率")
plt.title("bigramの確率表（一部）")
plt.show()

## 表から文章を作る


In [ ]:
g = torch.Generator().manual_seed(0)
ix = stoi["私"]
out = ["私"]
for _ in range(60):
    ix = torch.multinomial(P[ix], num_samples=1, generator=g).item()
    out.append(itos[ix])
print("".join(out))

## one-hotと行列の掛け算


In [ ]:
import torch.nn.functional as F

x = F.one_hot(torch.tensor(stoi["私"]), num_classes=V).float()
print(x.shape, x.sum())

row = x @ P            # one-hot × 確率表
print(row.shape)
print(torch.allclose(row, P[stoi["私"]]))

In [ ]:
A = torch.tensor([[1., 2.]])
B = torch.tensor([[3., 4., 5.],
                  [6., 7., 8.]])
print(A @ B)
print(A.shape, "@", B.shape, "→", (A @ B).shape)

## GPUは何が速いのか


In [ ]:
X = torch.randn(2000, 2000)
start = time.time(); _ = X @ X; print(f"CPU：{time.time() - start:.3f}秒")
if torch.cuda.is_available():
    Xg = X.to("cuda"); _ = Xg @ Xg; torch.cuda.synchronize()
    start = time.time(); _ = Xg @ Xg; torch.cuda.synchronize()
    print(f"GPU：{time.time() - start:.4f}秒")